<a href="https://colab.research.google.com/github/saisathwik2703/flyrank-ml-internship-starter/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/saisathwik2703/flyrank-ml-internship-starter/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [5]:
!git clone https://github.com/flyrank-bih/flyrank-ml-internship-starter.git /content/flyrank-ml-internship-starter

fatal: destination path '/content/flyrank-ml-internship-starter' already exists and is not an empty directory.


In [6]:
from pathlib import Path

DATA_PATH = Path(
    "/content/flyrank-ml-internship-starter/data/raw/content_refresh_anonymized.csv"
)

print("Dataset exists:", DATA_PATH.exists())
print("Dataset path:", DATA_PATH)

Dataset exists: True
Dataset path: /content/flyrank-ml-internship-starter/data/raw/content_refresh_anonymized.csv


In [7]:
import pandas as pd

df = pd.read_csv(
    "/content/flyrank-ml-internship-starter/data/raw/content_refresh_anonymized.csv"
)

print("Rows:", len(df))
print("Columns:", df.shape[1])
print("Shape:", df.shape)

Rows: 30000
Columns: 44
Shape: (30000, 44)


## My lane as an ML task

**Provisional lane:** Lane 4 — CTR / Engagement Opportunity Scoring.

**ML task type:** Ranking / scoring.

The goal is to rank pages by their potential CTR opportunity so that a content or SEO reviewer can start with the highest-priority pages. The output is an opportunity score rather than a simple yes/no decision because pages can have different levels of potential opportunity.

This is a decision-support task: the ranking helps a reviewer decide which pages to inspect first. It does not automatically decide that a page needs to be changed.

In [8]:
# Check that the main fields needed for Lane 4 exist.

required_columns = [
    "ctr",
    "impressions_90d",
    "position_tier"
]

missing_columns = [
    column for column in required_columns
    if column not in df.columns
]

print("Required columns available:", len(missing_columns) == 0)

if missing_columns:
    print("Missing columns:", missing_columns)
else:
    print("All required Lane 4 fields are available.")

Required columns available: True
All required Lane 4 fields are available.


## Target or proxy

The target for the first version is a **CTR opportunity proxy**.

I define a page as a proxy opportunity when its CTR is below the median CTR of other pages in the same position tier. This makes the comparison position-aware because pages in different search positions naturally have different expected CTR levels.

The proxy target is:

- `1` = page CTR is below the median CTR for its position tier.
- `0` = page CTR is at or above the median CTR for its position tier.

This is only a starting proxy. It does not mean that low CTR proves a page has poor content, metadata, or search intent. For a later model, I would prefer a future outcome such as whether CTR improves after review or intervention.

In [9]:
# Create the Lane 4 analysis slice.
# We use at least 100 impressions to reduce extremely noisy CTR estimates.

lane_df = df[df["impressions_90d"] >= 100].copy()

print(f"Pages with at least 100 impressions: {len(lane_df):,}")

# Calculate the median CTR within each position tier.
lane_df["tier_median_ctr"] = (
    lane_df.groupby("position_tier")["ctr"]
    .transform("median")
)

# Position-relative CTR gap.
lane_df["ctr_gap_proxy"] = (
    lane_df["tier_median_ctr"] - lane_df["ctr"]
).clip(lower=0)

# Proxy target.
lane_df["ctr_opportunity_proxy"] = (
    lane_df["ctr"] < lane_df["tier_median_ctr"]
).astype(int)

print("\nProxy target created successfully.")

display(
    lane_df[
        [
            "position_tier",
            "ctr",
            "tier_median_ctr",
            "ctr_gap_proxy",
            "ctr_opportunity_proxy"
        ]
    ].head(10)
)


Pages with at least 100 impressions: 22,006

Proxy target created successfully.


,position_tier,ctr,tier_median_ctr,ctr_gap_proxy,ctr_opportunity_proxy
0,striking,0.76,0.15,0.00,0
1,page_3_5,0.05,0.06,0.01,1
2,page_3_5,0.09,0.06,0.00,0
3,page_1,0.49,0.23,0.00,0
4,page_3_5,0.13,0.06,0.00,0
5,page_1,0.03,0.23,0.20,1
7,page_3_5,0.06,0.06,0.00,0
8,page_3_5,0.09,0.06,0.00,0
9,page_1,0.16,0.23,0.07,1
10,top_3,1.55,0.19,0.00,0


In [10]:
print("CTR opportunity proxy distribution:")

print(
    lane_df["ctr_opportunity_proxy"]
    .value_counts()
    .sort_index()
)

print("\nOpportunity percentage:")

print(
    lane_df["ctr_opportunity_proxy"].mean() * 100,
    "%"
)

CTR opportunity proxy distribution:
ctr_opportunity_proxy
0    11699
1    10307
Name: count, dtype: int64

Opportunity percentage:
46.837226211033354 %


## Success metric

The primary success metric will be **Precision@50**.

Precision@50 measures the proportion of true opportunity pages among the first 50 pages in the ranked review queue.

This metric fits the real decision because a reviewer has limited time. The goal is not simply to classify every page correctly. The goal is to put useful opportunities near the top of the queue.

I will compare the ML ranking with a simple baseline. A useful model should improve the quality of the top-ranked review queue on held-out data.

In [11]:
# Precision@K helper function.
# This will be used later when we have model scores.

def precision_at_k(y_true, scores, k=50):
    y_true = pd.Series(y_true).reset_index(drop=True)
    scores = pd.Series(scores).reset_index(drop=True)

    top_k_indices = scores.sort_values(
        ascending=False
    ).index[:k]

    return y_true.iloc[top_k_indices].mean()


print("Primary success metric: Precision@50")
print("Higher Precision@50 means a better top-50 review queue.")


Primary success metric: Precision@50
Higher Precision@50 means a better top-50 review queue.


## The unit of analysis, as a real dataframe

**Unit of analysis:** one row = one page observation.

Each page receives an opportunity score and can potentially enter the prioritized review queue. The page-level data contains CTR, impressions, position tier, and other measurable page/search characteristics.

The analysis therefore ranks individual page observations rather than clients, days, or search queries.

In [12]:
# Show the actual page-level dataframe used for the Lane 4 task.

available_columns = [
    column for column in [
        "page_id",
        "position_tier",
        "ctr",
        "impressions_90d",
        "avg_position",
        "content_type",
        "content_age_days",
        "word_count",
        "ctr_gap_proxy",
        "ctr_opportunity_proxy"
    ]
    if column in lane_df.columns
]

print("Unit of analysis: one row = one page observation")
print(f"Rows in Lane 4 slice: {len(lane_df):,}")

display(
    lane_df[available_columns].head(10)
)


Unit of analysis: one row = one page observation
Rows in Lane 4 slice: 22,006


,position_tier,ctr,impressions_90d,avg_position,content_type,content_age_days,word_count,ctr_gap_proxy,ctr_opportunity_proxy
0,striking,0.76,3803,10.6,keyword article,187,3221.0,0.00,0
1,page_3_5,0.05,15320,20.3,keyword article,445,2481.0,0.01,1
2,page_3_5,0.09,12581,36.5,keyword article,141,3515.0,0.00,0
3,page_1,0.49,11751,6.2,keyword article,463,NaN,0.00,0
4,page_3_5,0.13,19140,44.0,keyword article,263,2803.0,0.00,0
5,page_1,0.03,3970,8.5,keyword article,147,3080.0,0.20,1
7,page_3_5,0.06,1724,21.2,keyword article,445,NaN,0.00,0
8,page_3_5,0.09,32574,46.0,keyword article,90,3807.0,0.00,0
9,page_1,0.16,1240,4.9,keyword article,257,NaN,0.07,1
10,top_3,1.55,20919,2.2,keyword article,329,NaN,0.00,0


## Why ML beats a fixed rule here

A fixed rule could flag every page whose CTR is below one global threshold. However, CTR depends strongly on search position, so the same CTR can mean different things for pages in different position tiers.

A ranking model can combine several non-leaking signals and produce a continuous opportunity score. This allows the system to prioritize pages instead of applying one hard cutoff.

The advantage of ML is therefore not that it automatically decides what content should be changed. Its value is that it can learn useful combinations of signals and produce a more flexible ranking that can be evaluated against a simple baseline.

The final output remains decision support for a human content or SEO reviewer.

In [13]:
# Simple baseline:
# rank pages only by their position-tier-relative CTR gap.

baseline = lane_df.copy()

baseline = baseline.sort_values(
    "ctr_gap_proxy",
    ascending=False
)

print("Simple baseline created.")
print("Baseline ranking uses only the position-relative CTR gap.")

display(
    baseline[
        [
            "position_tier",
            "ctr",
            "tier_median_ctr",
            "ctr_gap_proxy",
            "ctr_opportunity_proxy"
        ]
    ].head(10)
)


Simple baseline created.
Baseline ranking uses only the position-relative CTR gap.


,position_tier,ctr,tier_median_ctr,ctr_gap_proxy,ctr_opportunity_proxy
11828,page_1,0.0,0.23,0.23,1
66,page_1,0.0,0.23,0.23,1
29913,page_1,0.0,0.23,0.23,1
29924,page_1,0.0,0.23,0.23,1
11909,page_1,0.0,0.23,0.23,1
11912,page_1,0.0,0.23,0.23,1
11916,page_1,0.0,0.23,0.23,1
11920,page_1,0.0,0.23,0.23,1
11921,page_1,0.0,0.23,0.23,1
11923,page_1,0.0,0.23,0.23,1


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.